# Convergent-Beam Electron Diffraction

A selected-area pattern is taken with a parallel beam, so every reflection is a spot: one
incident direction, one point of the rocking curve, one number per reflection. Converge the
beam instead — focus it to a probe of semi-angle $\alpha$, a few milliradians — and every spot
spreads into a **disc** whose every point corresponds to a different incident direction, and
therefore to a different deviation from the Bragg condition.

That is the whole idea. **One CBED exposure contains the rocking curve**, which a parallel
beam could only obtain by tilting the specimen through a series of exposures.

Three things follow, and this tutorial covers the first two quantitatively:

1. **Local thickness.** The rocking curve has zeros at deviations that depend on the foil
   thickness. Reading the fringe positions off one disc gives the thickness — and, from the
   same fit, the extinction distance, so the answer does not rest on a tabulated constant.
2. **The lattice repeat along the beam.** Convergence also makes higher-order Laue zones
   visible as rings, and their radii measure the one lattice dimension a zone-axis pattern is
   completely blind to.
3. **Point-group determination**, including whether the crystal has a centre of symmetry —
   which Friedel's law hides from kinematic SAED. That needs full dynamical intensities and is
   **not** implemented in PyTex; section 8 says so plainly rather than gesturing at it.

Both canonical cases run throughout: **nickel**, face-centred cubic, and **zirconium**,
hexagonal close-packed.

## 0. Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from pytex import (
    AtomicSite,
    ConvergentBeamConfig,
    Lattice,
    Phase,
    SpaceGroupSpec,
    SymmetrySpec,
    UnitCell,
    ZoneAxis,
    crystal_frame,
    extinction_distance_angstrom,
    fringe_minimum_excitation_errors,
    get_phase_fixture,
    holz_ring_radii_inv_angstrom,
    simulate_cbed_pattern,
    thickness_from_fringe_minima,
    two_beam_rocking_curve,
)
from pytex.diffraction.kinematic import electron_wavelength_angstrom

CRYSTAL = crystal_frame()

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz.*")
    warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF.*")
    NICKEL = get_phase_fixture("ni_fcc").load_phase(crystal_frame=CRYSTAL)
    ZIRCONIUM = get_phase_fixture("zr_hcp").load_phase(crystal_frame=CRYSTAL)

BEAM_KEV = 200.0
WAVELENGTH = electron_wavelength_angstrom(BEAM_KEV)

NI_ZONE = ZoneAxis(np.array([0, 0, 1]), phase=NICKEL)
ZR_ZONE = ZoneAxis(np.array([0, 0, 1]), phase=ZIRCONIUM)

print(f"beam {BEAM_KEV:.0f} kV, wavelength {WAVELENGTH:.6f} A")
for phase in (NICKEL, ZIRCONIUM):
    print(f"{phase.name:<15} a = {phase.lattice.a:.4f} A  c = {phase.lattice.c:.4f} A")

## 1. The extinction distance, and getting the scale right

Everything dynamical is measured in units of the **extinction distance**

$$\xi_g = \frac{\pi V_c \cos\theta_B}{\lambda\,|F_g|},$$

the thickness over which the intensity swings completely from the transmitted beam into the
diffracted beam and back. Getting $\xi_g$ right means getting $F_g$ onto an *absolute* scale,
in units of length, and this is the step at which a plausible but wrong CBED simulation is
easiest to produce: an extinction distance wrong by a constant gives perfectly convincing
fringes at the wrong spacing.

Two ingredients, both easy to get wrong and neither optional:

**Mott–Bethe.** Electrons are scattered by the electrostatic potential, not the charge
density, so the electron scattering factor comes from the X-ray form factor as

$$f_e(s) = \frac{Z - f_x(s)}{8\pi^2 a_0\, s^2} = \frac{Z - f_x(s)}{41.78214\, s^2},
\qquad s = \frac{\sin\theta}{\lambda} = \frac{|g|}{2},$$

in ångström. The numerator $Z - f_x$ is the nuclear charge *screened* by the electron cloud,
which is what the incident electron sees.

**The relativistic factor** $\gamma = 1 + E/m_0c^2$, which is 1.39 at 200 kV. Omitting it
would make every extinction distance 39 percent too long, and nothing else in the pattern
would look wrong.

In [ ]:
# Aluminium, built inline, is the calibration case: the fitted scattering-factor
# parametrization is most accurate for light elements.
def fcc_phase(name, species, parameter):
    lattice = Lattice(parameter, parameter, parameter, 90.0, 90.0, 90.0, crystal_frame=CRYSTAL)
    sites = tuple(
        AtomicSite(label=f"{species}{index}", species=species,
                   fractional_coordinates=np.asarray(position, dtype=float))
        for index, position in enumerate(
            [(0, 0, 0), (0, 0.5, 0.5), (0.5, 0, 0.5), (0.5, 0.5, 0)])
    )
    return Phase(name, lattice=lattice,
                 symmetry=SymmetrySpec.from_point_group("m-3m", reference_frame=CRYSTAL),
                 crystal_frame=CRYSTAL, unit_cell=UnitCell(lattice=lattice, sites=sites),
                 space_group=SpaceGroupSpec(symbol="Fm-3m", number=225, reference_frame=CRYSTAL))


ALUMINIUM = fcc_phase("aluminium-fcc", "Al", 4.0495)
PUBLISHED_AL_100KV = {(1, 1, 1): 556.0, (2, 0, 0): 673.0, (2, 2, 0): 1057.0}

print("Aluminium at 100 kV, against Williams & Carter (2nd ed.) Table 23.1:")
print(f"  {'hkl':<10} {'PyTex (A)':>10} {'published':>10} {'error':>8}")
for hkl, published in PUBLISHED_AL_100KV.items():
    computed = float(extinction_distance_angstrom(ALUMINIUM, hkl, beam_energy_kev=100.0)[0])
    print(f"  {str(hkl):<10} {computed:10.1f} {published:10.1f} "
          f"{100 * (computed / published - 1.0):7.1f}%")

In [ ]:
print(f"\nAt {BEAM_KEV:.0f} kV, for the two working cases:")
print(f"  {'phase':<15} {'hkl':<10} {'d (A)':>8} {'xi_g (A)':>10}")
for phase, reflections in ((NICKEL, [(1, 1, 1), (2, 0, 0), (2, 2, 0)]),
                           (ZIRCONIUM, [(1, 0, 0), (0, 0, 2), (1, 0, 1), (1, 1, 0)])):
    reciprocal = np.asarray(phase.lattice.reciprocal_basis().matrix)
    for hkl in reflections:
        spacing = 1.0 / np.linalg.norm(reciprocal @ np.asarray(hkl, dtype=float))
        xi = float(extinction_distance_angstrom(phase, hkl, beam_energy_kev=BEAM_KEV)[0])
        print(f"  {phase.name:<15} {str(hkl):<10} {spacing:8.4f} {xi:10.1f}")

Aluminium agrees within 1.4 percent. For nickel the same calculation runs about 11 percent
high against the usual tables: the fitted parametrization loses accuracy at small $s$ for
heavier elements. That limitation is stated in the docstring rather than hidden — and it is
exactly why quantitative CBED **measures** the extinction distance from the fringes rather
than taking it from a table, which is what section 5 does.

## 2. Disc geometry, and the two regimes

A tilt $\boldsymbol{\theta}$ of the incident direction displaces the diffracted beam by
$\boldsymbol{\theta}/\lambda$ in reciprocal space, so a cone of semi-angle $\alpha$ spreads
each reflection into a disc of reciprocal-space radius $\alpha/\lambda$. On a detector
calibrated by the camera constant $C = L\lambda$,

$$R_{\text{disc}} = C\,\frac{\alpha}{\lambda} = L\alpha .$$

The wavelength cancels: disc size on the plate is set by camera length and convergence angle
alone.

Neighbouring discs touch when $2\alpha/\lambda = |\mathbf{g}|_{\min}$, and that threshold
splits CBED into its two classical regimes:

- **Kossel–Möllenstedt**, $\alpha$ small: discs separated, each readable as an independent
  rocking curve. This is the regime thickness measurement needs.
- **Kossel**, $\alpha$ large: discs overlap, and the overlap regions carry interference
  between beams rather than a single rocking curve.

Note the threshold in another form: $2\alpha/\lambda = |\mathbf{g}|_{\min}$ is
$\alpha = \theta_B$ of the innermost reflection. So the discs touch exactly when the
convergence reaches the Bragg angle — which has a consequence we meet in section 5.

In [ ]:
def pattern_for(phase, zone, *, alpha_mrad, thickness_angstrom=1500.0, samples=241):
    return simulate_cbed_pattern(
        phase, zone,
        config=ConvergentBeamConfig(
            beam_energy_kev=BEAM_KEV,
            convergence_semi_angle_mrad=alpha_mrad,
            thickness_angstrom=thickness_angstrom,
            disc_samples=samples,
        ),
    )


def draw_pattern(ax, pattern, title):
    radius = pattern.config.disc_radius_mm
    for disc in pattern.discs:
        image = np.ma.masked_invalid(disc.intensity.T)
        centre = disc.centre_mm
        ax.imshow(image, origin="lower", cmap="inferno", vmin=0.0, vmax=1.0,
                  extent=(centre[0] - radius, centre[0] + radius,
                          centre[1] - radius, centre[1] + radius))
    centres = np.stack([disc.centre_mm for disc in pattern.discs])
    extent = np.abs(centres).max() + 2.0 * radius
    ax.set_xlim(-extent, extent); ax.set_ylim(-extent, extent)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{title}\n{pattern.config.convergence_semi_angle_mrad:g} mrad "
                 f"({pattern.regime})", fontsize=9)


fig, axes = plt.subplots(2, 3, figsize=(11.5, 7.8))
for row, (phase, zone, label) in enumerate(
    ((NICKEL, NI_ZONE, "Ni [001]"), (ZIRCONIUM, ZR_ZONE, "Zr [0001]"))
):
    for column, alpha in enumerate((3.0, 6.0, 11.0)):
        draw_pattern(axes[row, column], pattern_for(phase, zone, alpha_mrad=alpha), label)
fig.suptitle("CBED discs: the convergence angle sets the disc size and the regime", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
print(f"{'phase':<15} {'alpha':>6} {'disc radius':>12} {'nearest sep':>12} {'regime':>22}")
for phase, zone in ((NICKEL, NI_ZONE), (ZIRCONIUM, ZR_ZONE)):
    for alpha in (2.0, 4.0, 6.0, 8.0, 11.0):
        pattern = pattern_for(phase, zone, alpha_mrad=alpha, samples=15)
        print(f"{phase.name:<15} {alpha:5.1f}  {pattern.config.disc_radius_mm:11.1f} mm "
              f"{pattern.nearest_disc_separation_mm:9.1f} mm {pattern.regime:>22}")

for phase, zone in ((NICKEL, NI_ZONE), (ZIRCONIUM, ZR_ZONE)):
    pattern = pattern_for(phase, zone, alpha_mrad=2.0, samples=15)
    threshold = (pattern.nearest_disc_separation_mm
                 / (2.0 * pattern.config.camera_constant_mm_angstrom) * WAVELENGTH * 1e3)
    print(f"\n{phase.name}: discs touch at alpha = {threshold:.2f} mrad")

Zirconium goes over to the Kossel regime at a smaller convergence angle than nickel, because
its innermost allowed reflection $\{10\bar{1}0\}$ has a larger $d$-spacing and therefore a
smaller $|\mathbf{g}|$. The regime is a property of the *crystal* as much as of the
instrument.

## 3. What varies across a disc

Let the zone axis point toward the gun and the incident direction be tilted by
$(\theta_x, \theta_y)$. The excitation error of reflection $\mathbf{g} = (g_u, g_v, g_z)$ in
the zone basis is

$$s_g(\boldsymbol{\theta}) = g_z - \theta_x g_u - \theta_y g_v
                              - \tfrac{1}{2}\lambda|\mathbf{g}|^2,$$

which reduces at zero tilt to the parallel-beam expression the SAED engine uses. Two things
are readable straight off it:

- $s_g$ is **linear** in the tilt with gradient $-\mathbf{g}_\perp$, so lines of constant
  $s_g$ are straight and perpendicular to $\mathbf{g}$. That is why Kossel–Möllenstedt fringes
  are straight parallel bars, and why they are oriented differently in each disc.
- The disc is **centred** at $s_g = -\lambda g^2/2$, not at zero.

In [ ]:
pattern = pattern_for(NICKEL, NI_ZONE, alpha_mrad=6.0, thickness_angstrom=1500.0, samples=241)
disc = pattern.disc_for((2, 0, 0))
axis = disc.tilt_axis_mrad

fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.1))
for ax, field, title, cmap in (
    (axes[0], disc.excitation_error_inv_angstrom, r"excitation error $s_g$ (1/A)", "coolwarm"),
    (axes[1], disc.intensity, "two-beam intensity", "inferno"),
):
    image = ax.imshow(np.ma.masked_invalid(field.T), origin="lower", cmap=cmap,
                      extent=(axis[0], axis[-1], axis[0], axis[-1]))
    ax.set_title(f"Ni (200) disc: {title}", fontsize=9)
    ax.set_xlabel(r"$\theta_x$ (mrad)"); ax.set_ylabel(r"$\theta_y$ (mrad)")
    fig.colorbar(image, ax=ax, shrink=0.82)

s_values, intensity = disc.radial_profile()
axes[2].plot(s_values, intensity, lw=1.4)
axes[2].axvline(0.0, color="grey", ls="--", lw=0.8)
axes[2].set_xlabel(r"$s_g$ (1/A)"); axes[2].set_ylabel("intensity")
axes[2].set_title("the rocking curve, read along $\\mathbf{g}$", fontsize=9)
plt.tight_layout()
plt.show()

print(f"g direction on the detector: {np.round(disc.g_detector_inv_angstrom, 4)} 1/A")
print(f"disc centred at s = {-0.5 * WAVELENGTH * np.linalg.norm(disc.g_detector_inv_angstrom)**2:.5f} 1/A")
print(f"s spans {s_values.min():+.5f} to {s_values.max():+.5f} 1/A across the disc")
print(f"extinction distance used: {disc.extinction_distance_angstrom:.1f} A")

The fringes run perpendicular to $\mathbf{g}$, exactly as the linearity predicts. And notice
the third panel: the whole disc lies on **one side** of $s_g = 0$. Section 5 explains why that
is not a simulation artefact.

## 4. The two-beam rocking curve

With only the transmitted and one diffracted beam coupled, the Howie–Whelan equations have a
closed solution (no absorption):

$$I_g(s) = \frac{\sin^2(\pi t\, s_{\text{eff}})}{(\xi_g s_{\text{eff}})^2},
\qquad s_{\text{eff}} = \sqrt{s^2 + \xi_g^{-2}},
\qquad I_0 = 1 - I_g .$$

At exact Bragg incidence $I_g = \sin^2(\pi t/\xi_g)$: the beams exchange intensity completely
as thickness increases — the Pendellösung oscillation — which is what makes $\xi_g$ a
measurable length. The kinematic approximation is the $\xi_g \to \infty$ limit, and it is
already poor once $t \gtrsim \xi_g/3$.

In [ ]:
xi = float(extinction_distance_angstrom(NICKEL, (2, 0, 0), beam_energy_kev=BEAM_KEV)[0])
s_grid = np.linspace(-0.02, 0.02, 4001)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
for thickness in (400.0, 800.0, 1600.0, 3200.0):
    axes[0].plot(s_grid, two_beam_rocking_curve(
        s_grid, thickness_angstrom=thickness, extinction_distance_angstrom=xi),
        lw=1.2, label=f"t = {thickness:.0f} A")
axes[0].set_xlabel(r"$s$ (1/A)"); axes[0].set_ylabel(r"$I_g$")
axes[0].set_title(f"Ni (200), $\\xi_g$ = {xi:.0f} A: more thickness, more fringes", fontsize=9)
axes[0].legend(fontsize=8)

thicknesses = np.linspace(0.0, 3.0 * xi, 600)
axes[1].plot(thicknesses / xi, np.square(np.sin(np.pi * thicknesses / xi)), lw=1.4)
axes[1].set_xlabel(r"$t / \xi_g$"); axes[1].set_ylabel(r"$I_g(s=0)$")
axes[1].set_title(r"Pendelloesung at exact Bragg: $\sin^2(\pi t/\xi_g)$", fontsize=9)
plt.tight_layout()
plt.show()

print("Fringe count across a fixed s window grows linearly with thickness:")
for thickness in (400.0, 800.0, 1600.0, 3200.0):
    curve = two_beam_rocking_curve(
        s_grid, thickness_angstrom=thickness, extinction_distance_angstrom=xi)
    minima = fringe_minimum_excitation_errors(s_grid, curve)
    print(f"  t = {thickness:6.0f} A -> {len(minima):3d} minima in |s| < 0.02 1/A")

## 5. Measuring the thickness

The minima of the rocking curve are at $t\,s_{\text{eff},n} = n$ for integer $n$. Substituting
$s_{\text{eff}}^2 = s^2 + \xi_g^{-2}$ and dividing by $n^2$,

$$\boxed{\;\left(\frac{s_n}{n}\right)^{\!2} = \frac{1}{t^2}
        - \frac{1}{\xi_g^{2}}\,\frac{1}{n^{2}}\;}$$

so plotting $(s_n/n)^2$ against $1/n^2$ gives a **straight line** whose intercept is $t^{-2}$
and whose slope is $-\xi_g^{-2}$. One least-squares fit returns both unknowns; the thickness
does not inherit the error of a tabulated extinction distance. This is the linearization of
Kelly *et al.* (1975).

> **Algorithm — two-beam CBED thickness**
>
> 1. Read the dark-fringe positions $s_n$ from one disc, on the branch showing more fringes.
> 2. Assign consecutive integers $n_0, n_0+1, \dots$, innermost first.
> 3. Fit $(s_n/n)^2$ against $1/n^2$; require intercept $>0$ and slope $<0$.
> 4. If $n_0$ is unknown, repeat over candidates and take the best $R^2$ among physical fits.

### Why the disc shows one branch

Section 2 noted that the discs touch when $\alpha = \theta_B$ of the innermost reflection,
and section 3 noted that a disc is centred at $s_g = -\lambda g^2/2$ and spans
$\pm\alpha|\mathbf{g}_\perp|$. Put those together: **exact Bragg lies inside a disc only when
$\alpha > \theta_B$** — which is precisely the condition for leaving the Kossel–Möllenstedt
regime. At a zone axis you therefore get one wing of the rocking curve, and that is not a
defect of the simulation.

It does not matter. The Kelly fit needs the *positions* of the minima and their orders, not
both wings. In real practice one tilts off the zone axis to a two-beam condition, which
recentres the disc; PyTex simulates zone axes, so the wing is what we read.

In [ ]:
def measure_thickness(phase, zone, hkl, *, alpha_mrad, thickness_angstrom, samples=1201):
    pattern = simulate_cbed_pattern(
        phase, zone,
        config=ConvergentBeamConfig(
            beam_energy_kev=BEAM_KEV, convergence_semi_angle_mrad=alpha_mrad,
            thickness_angstrom=thickness_angstrom, disc_samples=samples,
        ),
    )
    disc = pattern.disc_for(hkl)
    s_values, intensity = disc.radial_profile()
    minima = fringe_minimum_excitation_errors(s_values, intensity)
    negative, positive = minima[minima < 0.0], minima[minima > 0.0]
    branch = negative if negative.size >= positive.size else positive
    return pattern, disc, branch, thickness_from_fringe_minima(branch)


CASES = [
    ("Ni (200)", NICKEL, NI_ZONE, (2, 0, 0), 6.0, 1500.0),
    ("Ni (200)", NICKEL, NI_ZONE, (2, 0, 0), 6.5, 2000.0),
    ("Zr (10-10)", ZIRCONIUM, ZR_ZONE, (1, 0, 0), 4.0, 2000.0),
    ("Zr (10-10)", ZIRCONIUM, ZR_ZONE, (1, 0, 0), 4.0, 3000.0),
]

print(f"{'case':<12} {'alpha':>6} {'regime':>21} {'fringes':>8} {'n0':>3} "
      f"{'t true':>8} {'t fit':>8} {'xi true':>8} {'xi fit':>8} {'R^2':>9}")
for label, phase, zone, hkl, alpha, thickness in CASES:
    pattern, disc, branch, report = measure_thickness(
        phase, zone, hkl, alpha_mrad=alpha, thickness_angstrom=thickness)
    print(f"{label:<12} {alpha:5.1f}  {pattern.regime:>21} {branch.size:8d} "
          f"{report.first_order:3d} {thickness:8.0f} {report.thickness_angstrom:8.1f} "
          f"{disc.extinction_distance_angstrom:8.1f} "
          f"{report.extinction_distance_angstrom:8.1f} {report.r_squared:9.6f}")

Every case recovers the thickness that went in, and the extinction distance alongside it, from
fringe positions alone. Both patterns stayed in the Kossel–Möllenstedt regime, where the
fringes mean what the fit assumes.

Here is the fit itself for one case — the straight line is the whole method.

In [ ]:
pattern, disc, branch, report = measure_thickness(
    ZIRCONIUM, ZR_ZONE, (1, 0, 0), alpha_mrad=4.0, thickness_angstrom=3000.0)

orders = np.asarray(report.orders, dtype=float)
abscissa = 1.0 / orders**2
ordinate = (np.abs(report.excitation_errors_inv_angstrom) / orders) ** 2
line_x = np.linspace(0.0, abscissa.max() * 1.05, 100)
slope = -1.0 / report.extinction_distance_angstrom**2
intercept = 1.0 / report.thickness_angstrom**2

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
s_values, intensity = disc.radial_profile()
axes[0].plot(s_values, intensity, lw=1.3)
axes[0].plot(-np.abs(branch), np.zeros_like(branch), "v", color="tab:red", ms=7,
             label="fringe minima")
axes[0].set_xlabel(r"$s$ (1/A)"); axes[0].set_ylabel("intensity")
axes[0].set_title(r"Zr $(10\bar{1}0)$ disc profile, $t$ = 3000 A", fontsize=9)
axes[0].legend(fontsize=8)

axes[1].plot(line_x, slope * line_x + intercept, "-", color="tab:grey", lw=1.2)
axes[1].plot(abscissa, ordinate, "o", color="tab:red")
axes[1].axhline(intercept, color="tab:blue", ls=":", lw=1.0)
axes[1].set_xlabel(r"$1/n^2$"); axes[1].set_ylabel(r"$(s_n/n)^2$")
axes[1].set_title(f"Kelly plot: intercept $1/t^2$ gives t = "
                  f"{report.thickness_angstrom:.0f} A,\nslope gives "
                  f"$\\xi_g$ = {report.extinction_distance_angstrom:.0f} A", fontsize=9)
axes[1].set_xlim(left=0.0)
plt.tight_layout()
plt.show()

print(report.describe())

## 6. The failure mode, deliberately triggered

The order assigned to the innermost *visible* minimum is the method's known weakness. It is
rarely $n = 1$ — the low-order minima fall outside the disc — and assuming so biases the
answer.

Two things happen when the assignment is wrong, and PyTex uses both.

In [ ]:
# A genuinely awkward case: a large convergence angle pushes the disc into the
# Kossel regime, where the fringes no longer belong to a single rocking curve.
pattern, disc, branch, searched = measure_thickness(
    ZIRCONIUM, ZR_ZONE, (1, 0, 0), alpha_mrad=10.0, thickness_angstrom=2500.0)

print(f"regime: {pattern.regime}")
print(f"true    : t = 2500 A, xi = {disc.extinction_distance_angstrom:.0f} A")
print(f"searched: t = {searched.thickness_angstrom:.0f} A, "
      f"xi = {searched.extinction_distance_angstrom:.0f} A, "
      f"n0 = {searched.first_order}, R^2 = {searched.r_squared:.5f}")
print("\nThe fit is visibly worse (R^2 well below the 0.99999 of section 5), and both")
print("numbers are off. The R^2 is the warning, and it is reported for that reason.")

In [ ]:
# And forcing the classic wrong assumption on a clean case does not merely bias
# the answer: it tilts the fitted line the wrong way, implying a negative
# 1/xi^2, so there is no thickness to report at all.
_, _, clean_branch, clean = measure_thickness(
    NICKEL, NI_ZONE, (2, 0, 0), alpha_mrad=6.5, thickness_angstrom=2000.0)
print(f"searched assignment: n0 = {clean.first_order}, t = {clean.thickness_angstrom:.0f} A")

try:
    thickness_from_fringe_minima(clean_branch, first_order=1)
except ValueError as error:
    print(f"\nforcing n0 = 1 raises rather than returning a plausible number:\n  {error}")

## 7. HOLZ rings: the lattice repeat along the beam

A zone-axis pattern is blind to the lattice repeat *along* the beam, because every
zeroth-Laue-zone reflection is perpendicular to the zone axis. Higher-order Laue zones restore
it. With $H = 1/|\mathbf{r}_{uvw}|$ the reciprocal-lattice layer spacing along the zone axis,
the Ewald sphere cuts the $n$-th layer on a circle of projected radius

$$G_n \simeq \sqrt{\frac{2nH}{\lambda}} .$$

Since $G_n \propto \sqrt{H}$, a ring radius converts straight back into the repeat distance —
and that is the basis of CBED lattice-parameter and strain metrology. The round trip below
recovers $a$ for nickel down $[001]$ and, more usefully, $c$ for zirconium down $[0001]$: the
$c$ parameter that the $[0001]$ zone-axis *spot* pattern cannot see at all.

In [ ]:
HOLZ_CASES = [
    ("Ni", NICKEL, (0, 0, 1), "a", NICKEL.lattice.a),
    ("Ni", NICKEL, (1, 1, 1), "a*sqrt(3)", NICKEL.lattice.a * np.sqrt(3.0)),
    ("Zr", ZIRCONIUM, (0, 0, 1), "c", ZIRCONIUM.lattice.c),
    ("Zr", ZIRCONIUM, (1, 1, 0), "a", ZIRCONIUM.lattice.a),
]

print(f"{'phase':<6} {'zone':<10} {'orders':<10} {'G_1 (1/A)':>10} "
      f"{'repeat (A)':>11} {'is':<10} {'expected':>10}")
for label, phase, uvw, name, expected in HOLZ_CASES:
    orders, radii = holz_ring_radii_inv_angstrom(
        phase, ZoneAxis(np.asarray(uvw), phase=phase), beam_energy_kev=BEAM_KEV)
    layer_spacing = WAVELENGTH * radii[0] ** 2 / (2.0 * orders[0])
    order_text = ", ".join(str(int(value)) for value in orders)
    print(f"{label:<6} {str(uvw):<10} {order_text:<10} {radii[0]:10.3f} "
          f"{1.0 / layer_spacing:11.4f} {name:<10} {expected:10.4f}")

Exact, because the layer spacing itself is exact — only the ring *radius* uses the small-angle
approximation, and inverting it here uses the same expression. In a real experiment the ring
radius is a measurement, and the accuracy of $c$ follows from the accuracy of that radius; the
point demonstrated is that the information is present in the pattern and recoverable in closed
form.

Note the ring radii: several inverse ångström, far outside the zeroth-order discs. HOLZ rings
need a much wider camera field than the spot pattern, which is why they are often simply not
recorded.

## 8. What this implementation does not do

Stated plainly, and repeated by `CBEDPattern.describe()` on every pattern, so nothing here can
be mistaken for a dynamical calculation.

- **Many-beam coupling.** Each disc is its own two-beam calculation, so the discs of one
  pattern are not mutually consistent and their *relative* intensities carry no information.
  At a zone axis — where many reflections are simultaneously excited — this is the two-beam
  picture at its weakest. Geometry, regime and fringe positions are meaningful; relative disc
  brightness is not.
- **Absorption.** The fringes therefore do not decay with thickness as they do in practice,
  and a real disc shows fewer usable fringes than these simulations do.
- **HOLZ lines** within the bright-field disc — the sharp features used for lattice-parameter
  metrology at part-per-thousand accuracy. Only the ring radii are computed.
- **Diffraction-group symmetry determination.** The symmetry within and between discs fixes
  the diffraction group and hence the point group *including* whether the crystal is
  centrosymmetric, which Friedel's law hides from kinematic SAED. That is arguably the most
  celebrated capability of CBED, it needs full dynamical intensities, and it is not attempted.

In [ ]:
print(pattern_for(ZIRCONIUM, ZR_ZONE, alpha_mrad=4.0, samples=15).describe())

## 9. What to take away

- **A CBED disc is a rocking curve.** That is the entire reason to converge the beam, and
  everything else follows from it.
- **Check the regime before reading fringes.** `is_kossel_moellenstedt` is a geometric fact
  about your convergence angle and your crystal, and outside it a disc is not a rocking curve.
- **Fit the thickness and the extinction distance together.** The Kelly line gives both, so
  the thickness does not depend on a tabulated $\xi_g$ — which matters, because the tabulated
  value is only good to a few percent for light elements and worse for heavy ones.
- **Watch $R^2$ and the fitted $n_0$.** They are the method's self-diagnosis; a wrong order
  assignment either curves the plot or makes it unphysical.
- **HOLZ rings carry the third lattice dimension.** For a hexagonal metal down $[0001]$ that
  is $c$, which the spot pattern cannot supply.

### Further reading

- `docs/tex/algorithms/convergent_beam_electron_diffraction.tex` — the derivations, including
  the disc geometry and the Mott–Bethe scale.
- Tutorial 27, *TEM diffraction pattern indexing* — the parallel-beam companion, and the
  ambiguities a single SAED pattern leaves.
- Tutorial 24, *TEM tilt navigation* — getting to a zone axis in the first place.
- Kelly, Jostsons, Blake and Napier, *Phys. Status Solidi (a)* **31** (1975) 771 — the
  thickness linearization.
- Williams and Carter, *Transmission Electron Microscopy*, 2nd ed., Chapters 21–23 — two-beam
  theory, extinction distances, and CBED practice.
- Steeds, in *Introduction to Analytical Electron Microscopy* (Plenum, 1979) — the
  diffraction-group method this implementation does not attempt.